In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## step1: loading data

In [3]:
features = pd.read_csv("../data/features.csv")
features['filing_date'] = pd.to_datetime(features['filing_date'])
print(features[['ticker', 'filing_date', 'composite_signal']].to_string(index=False))

ticker filing_date  composite_signal
  AAPL  2025-07-31         -0.028428
  AAPL  2025-10-30         -0.634671
  AAPL  2026-01-29         -0.246660
  AMZN  2025-07-31          1.202853
  AMZN  2025-10-30          1.544172
  AMZN  2026-02-05         -0.264927
 GOOGL  2025-10-29          0.459914
 GOOGL  2026-02-04         -0.622486
  META  2025-07-30          0.783658
  META  2025-10-29          0.380878
  META  2026-01-28         -1.575917
  MSFT  2025-04-30         -0.142280
  MSFT  2025-07-30         -0.227171
  MSFT  2025-10-29         -0.147725
  MSFT  2026-01-28         -0.481209


## step2: downlaoding Fama-French factors

In [5]:
import urllib.request
import zipfile
import io

url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip"

print("Downloading Fama-French factors...")
with urllib.request.urlopen(url) as response:
    zip_data = response.read()

with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
    print("files in zip:", z.namelist())
    fname = z.namelist()[0]
    with z.open(fname) as f:
        raw = f.read().decode('utf-8')

files in zip: ['F-F_Research_Data_Factors_daily.csv']


In [6]:
lines = raw.strip().split('\n')
start = next(i for i, l in enumerate(lines) if l.strip().startswith('19') or l.strip().startswith('20'))
ff_df = pd.read_csv(io.StringIO('\n'.join(lines[start:])), header=None, names=['date', 'mkt_rf', 'smb', 'hml', 'rf'])

ff_df['date'] = pd.to_datetime(ff_df['date'].astype(str).str.strip(), format='%Y%m%d', errors='coerce')
ff_df[['mkt_rf', 'smb', 'hml', 'rf']] = ff_df[['mkt_rf', 'smb', 'hml', 'rf']].apply(pd.to_numeric, errors='coerce') / 100
ff_df = ff_df.dropna().set_index('date')

print(f"Fama-French factors loaded: {len(ff_df)} trading days")
print(ff_df.tail(3))

Fama-French factors loaded: 26190 trading days
            mkt_rf     smb     hml      rf
date                                      
2026-02-25  0.0079 -0.0037  0.0049  0.0001
2026-02-26 -0.0047  0.0063  0.0032  0.0001
2026-02-27 -0.0051 -0.0044 -0.0125  0.0001


## step3: downloading stock prices for event windows

In [7]:
TICKERS = ["AAPL", "MSFT", "GOOGL", "META", "AMZN"]

# 3 years of daily prices
prices = yf.download(TICKERS, start="2023-01-01", end="2026-04-01", auto_adjust=True)['Close']
prices.index = pd.to_datetime(prices.index)
# removing timezone
prices.index = prices.index.tz_localize(None) if prices.index.tz else prices.index

print(f"Price data: {prices.shape}")
print(prices.tail(3))

[*********************100%***********************]  5 of 5 completed

Price data: (813, 5)
Ticker            AAPL        AMZN       GOOGL        META        MSFT
Date                                                                  
2026-03-27  248.800003  199.339996  274.339996  525.719971  356.769989
2026-03-30  246.630005  200.949997  273.500000  536.380005  358.959991
2026-03-31  253.789993  208.270004  287.559998  572.130005  370.170013


## step4: computing daily returns

In [8]:
returns = prices.pct_change().dropna()
print(f"returns computed: {returns.shape}")
print(returns.tail(3))

returns computed: (812, 5)
Ticker          AAPL      AMZN     GOOGL      META      MSFT
Date                                                        
2026-03-27 -0.016173 -0.039510 -0.023423 -0.039851 -0.025139
2026-03-30 -0.008722  0.008077 -0.003062  0.020277  0.006138
2026-03-31  0.029031  0.036427  0.051408  0.066651  0.031229


## step5: computing abnormal returns for each earnings event

In [9]:
def compute_car(ticker, event_date, returns, ff_df, window=(-1, 3)):
    ticker_returns = returns[ticker].dropna() # get trading days

    # find event date index
    trading_days = ticker_returns.index
    event_matches = trading_days[trading_days >= event_date]
    if len(event_matches) == 0:
        return np.nan

    event_idx = trading_days.get_loc(event_matches[0])

    # estimation window - 130 to 11 days before the event
    est_start = max(0, event_idx - 130)
    est_end = max(0, event_idx - 11)

    if est_end - est_start < 30:
        return np.nan
    
    est_returns = ticker_returns.iloc[est_start:est_end]
    est_dates = est_returns.index
    
    # aligning with FF factors
    ff_est = ff_df.reindex(est_dates).dropna()
    est_returns_aligned = est_returns.reindex(ff_est.index)
    
    # OLS: stock return = alpha + (beta * MktRf) + (beta * SMB) + (beta * HML)
    X = sm.add_constant(ff_est[['mkt_rf', 'smb', 'hml']])
    y = est_returns_aligned - ff_est['rf']
    
    try:
        model = sm.OLS(y, X).fit()
    except Exception:
        return np.nan
    
    # event window returns
    ev_start = event_idx + window[0]
    ev_end = event_idx + window[1] + 1
    ev_returns = ticker_returns.iloc[max(0, ev_start):min(len(trading_days), ev_end)]
    ev_dates = ev_returns.index
    ff_ev = ff_df.reindex(ev_dates).dropna()
    ev_returns_aligned = ev_returns.reindex(ff_ev.index)
    
    # expected returns
    X_ev = sm.add_constant(ff_ev[['mkt_rf', 'smb', 'hml']], has_constant='add')
    expected = model.predict(X_ev)
    
    # abnormal returns
    abnormal = (ev_returns_aligned - ff_ev['rf'].values) - expected.values
    car = abnormal.sum()
    
    return float(car)

## step6: computing CAR for all events

In [11]:
cars = []

for idx, row in features.iterrows():
    ticker = row['ticker']
    event_date = row['filing_date']
    
    car = compute_car(ticker, event_date, returns, ff_df)
    cars.append(car)
    print(f"{ticker} | {event_date.date()} | CAR = {car:.4f}" if not np.isnan(car) else f"{ticker} | {event_date.date()} | CAR = NaN")

features['CAR'] = cars

AAPL | 2025-07-31 | CAR = -0.0149
AAPL | 2025-10-30 | CAR = 0.0231
AAPL | 2026-01-29 | CAR = 0.0425
AMZN | 2025-07-31 | CAR = -0.0674
AMZN | 2025-10-30 | CAR = 0.1212
AMZN | 2026-02-05 | CAR = -0.1405
GOOGL | 2025-10-29 | CAR = 0.0415
GOOGL | 2026-02-04 | CAR = -0.0595
META | 2025-07-30 | CAR = 0.0799
META | 2025-10-29 | CAR = -0.1562
META | 2026-01-28 | CAR = 0.0767
MSFT | 2025-04-30 | CAR = 0.0891
MSFT | 2025-07-30 | CAR = 0.0258
MSFT | 2025-10-29 | CAR = -0.0359
MSFT | 2026-01-28 | CAR = -0.0897


## step7: running OLS regression to check if our signal predicts CAR

In [12]:
df_reg = features.dropna(subset=['CAR', 'composite_signal', 'tone_surprise_z', 'earnings_surprise_z']).copy()

# regression: CAR ~ composite_signal
X = sm.add_constant(df_reg[['composite_signal', 'tone_surprise_z', 'earnings_surprise_z']])
y = df_reg['CAR']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    CAR   R-squared:                       0.232
Model:                            OLS   Adj. R-squared:                  0.104
Method:                 Least Squares   F-statistic:                     1.811
Date:                Sat, 11 Apr 2026   Prob (F-statistic):              0.205
Time:                        20:37:15   Log-Likelihood:                 18.191
No. Observations:                  15   AIC:                            -30.38
Df Residuals:                      12   BIC:                            -28.26
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.0043    

### caveats:
1. **small sample**: 15 observations is below the (atleast) 30 needed for reliable OLS inference. so these results should be interpreted as directional, and are in no way conclusive.
2. **multicollinearity**: `composite_signal` is constructed from `tone_surprise_z` and `earnings_surprise_z` so they are correlated by design. a cleaner spec would use just the two components separately.
3. **no transaction costs**: abnormal returns are gross - real trading would incur bid-ask spreads and market impact.
4. **limited universe**: have only taken the data for 5 mega-cap tech companies over <1 year. so, results may not generalize to other sectors or market regimes.

- also, the tone surprise signal (`tone_surprise_z` - it is in the expected direction but not at all significant at convetional levels) means there should be further investigation with a larger dataset

## step8: T-test for high signal vs low signal

In [14]:
median_signal = df_reg['composite_signal'].median()
high_signal = df_reg[df_reg['composite_signal'] > median_signal]['CAR']
low_signal = df_reg[df_reg['composite_signal'] <= median_signal]['CAR']

t_stat, p_value = stats.ttest_ind(high_signal, low_signal)

print(f"high signal group:")
print(f"mean CAR: {high_signal.mean():.4f}")
print()
print(f"low signal group:")
print(f"mean CAR: {low_signal.mean():.4f}")
print()
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print()

high signal group:
mean CAR: 0.0133

low signal group:
mean CAR: -0.0197

T-statistic: 0.7385
P-value: 0.4733



### caveat:
- since, the p-value is >0.1, it is not statistically significant, but again, this was expected coz there are only 15 events

## saving the CAR data

In [15]:
df_reg.to_csv("../data/car_results.csv", index=False)
print(df_reg[['ticker', 'filing_date', 'composite_signal', 'CAR']].to_string(index=False))

ticker filing_date  composite_signal       CAR
  AAPL  2025-07-31         -0.028428 -0.014939
  AAPL  2025-10-30         -0.634671  0.023067
  AAPL  2026-01-29         -0.246660  0.042500
  AMZN  2025-07-31          1.202853 -0.067367
  AMZN  2025-10-30          1.544172  0.121244
  AMZN  2026-02-05         -0.264927 -0.140464
 GOOGL  2025-10-29          0.459914  0.041535
 GOOGL  2026-02-04         -0.622486 -0.059498
  META  2025-07-30          0.783658  0.079913
  META  2025-10-29          0.380878 -0.156172
  META  2026-01-28         -1.575917  0.076662
  MSFT  2025-04-30         -0.142280  0.089116
  MSFT  2025-07-30         -0.227171  0.025766
  MSFT  2025-10-29         -0.147725 -0.035870
  MSFT  2026-01-28         -0.481209 -0.089686
